# OpenChat V2 — Run on Google Colab + Cloudflare Tunnel

This notebook builds the app, serves it on port 3000 with `server.js` (Express — static build + `/api/auth/*` backed by `accounts.json`), and exposes it via a free `trycloudflare.com` quick tunnel.

**Project zip is fetched automatically from catbox** (https://files.catbox.moe/2063uv.zip).

Run the cells top to bottom.

In [ ]:
# 1. Download the project zip from catbox
!wget -q https://files.catbox.moe/2063uv.zip -O openchat-v2.zip
!ls -lh openchat-v2.zip

In [ ]:
# 2. Unzip and enter the project
import zipfile, os
with zipfile.ZipFile('openchat-v2.zip') as z:
    z.extractall('/content/openchat-v2')
project = '/content/openchat-v2'
print('Project at:', project)
%cd {project}

In [ ]:
# 3. Install dependencies
!npm install

In [ ]:
# 4. (Optional) Set your Gemini API key if the app needs it
import os
os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'  # or use Colab Secrets (key icon) and:
# from google.colab import userdata
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
# 5. Build production bundle
!npm run build

In [ ]:
# 6. Start the server on port 3000 in the background#    (serves dist/ + /api/auth/*; accounts persist in /content/openchat-v2/accounts.json)
import subprocess, time, os
env = dict(os.environ, PORT='3000')
proc = subprocess.Popen(
    ['node', 'server.js'],
    env=env,
    stdout=open('/content/preview.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(3)
!curl -s -o /dev/null -w 'preview HTTP status: %{http_code}\n' http://localhost:3000

In [ ]:
# 7. Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 8. Start the quick tunnel and print the public URL
import subprocess, re, time
tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    time.sleep(2)
    try:
        text = open('/content/tunnel.log').read()
    except Exception:
        continue
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        url = m.group(0)
        break
print('🌐 Public URL:', url or 'Not ready yet — check /content/tunnel.log')

Keep the runtime alive — closing the Colab tab or disconnecting stops the tunnel.
To stop: `tunnel.terminate()` and `proc.terminate()`.